In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("kronomy/helmet-dataset-by-osf-lite")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/kronomy/helmet-dataset-by-osf-lite


# Motorcycle Helmet Detection — Bước 3: Tracking (BoT-SORT)

## Tổng quan
Notebook này thuộc **Luồng B — INFERENCE**, xây dựng inference engine kết hợp YOLO + BoT-SORT để:
1. Phát hiện xe máy trên từng frame (dùng `best.pt` từ Bước 2)
2. Gán và theo dõi Track ID cho từng xe máy qua các frame
3. Thu thập crop ảnh xe máy theo từng track
4. Xuất output để dùng ở Bước 5 (voting + pipeline hoàn chỉnh)

## Input (từ Kaggle Dataset)
- **Dataset gốc**: `/kaggle/input/helmet-dataset-by-osf-lite/helmet-dataset` (ảnh gốc)
  - Hoặc theo đường dẫn thực tế trên Kaggle dataset của bạn
- **Weights Bước 2**: `/kaggle/input/helmet-step1-2-outputs/runs/helmet_yolo/weights/best.pt`
  - Cần upload kết quả Bước 1 & 2 lên Kaggle Dataset rồi thêm làm input

## Output (vào `/kaggle/working/`)
- `track_crops/{video_id}/track_{id}/frame_{frame:06d}.jpg` — crops theo Track ID
- `track_metadata.json` — metadata từng track
- `step3_tracking_summary.json` — thống kê tổng

## Lưu ý quan trọng
- **Không dùng Ground Truth annotation** — đây là inference thuần túy
- Mỗi clip phải reset tracker để tránh nhầm lẫn giữa các clip
- Output sẽ được dùng ở Bước 5 (KHÔNG phải Bước 4)
- Bước 4 (train EfficientNetV2) dùng crops từ Bước 1 (Ground Truth)

## Vị trí trong pipeline
```
LUỒNG A (Training): Bước 1 → Bước 4 (EfficientNetV2)
LUỒNG B (Inference): Bước 2 (YOLO weights) + Bước 3 (BoT-SORT) → Bước 5
                                          ↓ hội tụ tại
                                        Bước 5
                                          ↓
                                        Bước 6 (Evaluation)
```

## Section 1: Cài đặt & Import thư viện

In [2]:
!pip install -U ultralytics lapx --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 68.5 MB/s eta 0:00:00


In [3]:
import os
import cv2
import json
import time
import shutil
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict

import torch
from ultralytics import YOLO

print(f"✓ PyTorch version   : {torch.__version__}")
print(f"✓ CUDA available    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  - GPU             : {torch.cuda.get_device_name(0)}")
    print(f"  - VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"✓ Ultralytics import: OK")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✓ PyTorch version   : 2.10.0+cu128
✓ CUDA available    : True
  - GPU             : Tesla T4
  - VRAM            : 15.6 GB
✓ Ultralytics import: OK


## Section 2: Cấu hình đường dẫn & tham số

In [4]:
# ============================================================================
# CONFIGURATION — Chỉnh sửa các đường dẫn cho phù hợp với Kaggle dataset của bạn
# ============================================================================

# --- Đường dẫn dataset gốc (ảnh frames) ---
# Thay thế bằng đường dẫn thực tế trên Kaggle của bạn
# Ví dụ: /kaggle/input/helmet-dataset-by-osf-lite/helmet-dataset
DATASET_ROOT = '/kaggle/input/datasets/kronomy/helmet-dataset-by-osf-lite/helmet-dataset'

# --- Đường dẫn output Bước 1 & 2 (đã upload lên Kaggle Dataset) ---
# Thay thế bằng tên dataset bạn đã tạo trên Kaggle
STEP12_INPUT = '/kaggle/input/datasets/uynguyenthai/helmet-step2-outputs'

# Đường dẫn weights YOLO tốt nhất từ Bước 2
BEST_WEIGHTS = os.path.join(STEP12_INPUT, 'best.pt')

# --- Output directory ---
OUTPUT_BASE  = '/kaggle/working'
TRACK_CROPS_DIR = os.path.join(OUTPUT_BASE, 'track_crops')   # crops theo track
METADATA_PATH   = os.path.join(OUTPUT_BASE, 'track_metadata.json')
SUMMARY_PATH    = os.path.join(OUTPUT_BASE, 'step3_tracking_summary.json')
BOTSORT_CFG     = os.path.join(OUTPUT_BASE, 'botsort_custom.yaml')

# --- Tham số tracking ---
TRACK_BUFFER      = 10    # 10 FPS → buffer 10 = 1 giây thực
MAX_CROPS_PER_TRACK = 80  # giới hạn số crop/track (100 frames/clip tối đa)
CROP_PADDING      = 0.05  # padding 5% cho mỗi phía khi crop
CROP_SIZE         = 224   # resize về 224×224 (chuẩn EfficientNetV2 input)
CONFIDENCE_THRESH = 0.25  # ngưỡng confidence cho YOLO detection
IOU_THRESH        = 0.45  # ngưỡng IoU cho NMS
IMG_SIZE          = 640   # kích thước ảnh inference YOLO

# --- Phân chia split cần chạy ---
# Chọn subset để chạy: 'all', 'train', 'val', 'test'
# Khuyến nghị: 'test' trước để kiểm tra, sau đó 'all'
RUN_SPLIT = 'test'  # <-- Thay đổi nếu cần

# --- Thông số ảnh gốc ---
IMAGE_HEIGHT, IMAGE_WIDTH = 1080, 1920

# Tạo thư mục output
os.makedirs(TRACK_CROPS_DIR, exist_ok=True)

print(f"✓ Cấu hình hoàn tất:")
print(f"  - Dataset root  : {DATASET_ROOT}")
print(f"  - YOLO weights  : {BEST_WEIGHTS}")
print(f"  - Output base   : {OUTPUT_BASE}")
print(f"  - Track crops   : {TRACK_CROPS_DIR}")
print(f"  - Track buffer  : {TRACK_BUFFER} frames")
print(f"  - Max crops/track: {MAX_CROPS_PER_TRACK}")
print(f"  - Run split     : {RUN_SPLIT}")

✓ Cấu hình hoàn tất:
  - Dataset root  : /kaggle/input/datasets/kronomy/helmet-dataset-by-osf-lite/helmet-dataset
  - YOLO weights  : /kaggle/input/datasets/uynguyenthai/helmet-step2-outputs/best.pt
  - Output base   : /kaggle/working
  - Track crops   : /kaggle/working/track_crops
  - Track buffer  : 10 frames
  - Max crops/track: 80
  - Run split     : test


## Section 3: Kiểm tra Input

In [5]:
# ============================================================================
# Kiểm tra tồn tại của các file/thư mục cần thiết
# ============================================================================

print("=" * 70)
print("KIỂM TRA INPUT")
print("=" * 70)

# 1. Kiểm tra dataset gốc
images_base = os.path.join(DATASET_ROOT, 'images')
ann_base    = os.path.join(DATASET_ROOT, 'annotation', 'annotation')
split_csv   = os.path.join(DATASET_ROOT, 'data_split.csv')

for path, label in [(DATASET_ROOT, 'Dataset root'), 
                     (images_base,  'Images base'),
                     (split_csv,    'data_split.csv')]:
    status = "✓" if os.path.exists(path) else "✗ KHÔNG TÌM THẤY"
    print(f"  {status}: {label} — {path}")

# 2. Kiểm tra YOLO weights
weights_ok = os.path.exists(BEST_WEIGHTS)
print(f"  {'✓' if weights_ok else '✗ KHÔNG TÌM THẤY'}: YOLO weights — {BEST_WEIGHTS}")
if not weights_ok:
    print()
    print("  ⚠️  CẢNH BÁO: Không tìm thấy YOLO weights!")
    print("  Hãy chắc chắn bạn đã:")
    print("  1. Upload output của Bước 2 lên Kaggle Dataset")
    print("  2. Thêm dataset đó vào Kaggle Notebook này")
    print("  3. Cập nhật đường dẫn STEP12_INPUT trong Section 2")
    raise FileNotFoundError(f"YOLO weights không tìm thấy tại: {BEST_WEIGHTS}")

# 3. Đọc data split
split_df = pd.read_csv(split_csv)
split_map = {'training': 'train', 'validation': 'val', 'test': 'test'}
split_df['split'] = split_df['Set'].map(split_map)

print(f"\n✓ Data split:")
print(f"  - Tổng clips : {len(split_df)}")
print(f"  - Train      : {len(split_df[split_df['split'] == 'train'])}")
print(f"  - Val        : {len(split_df[split_df['split'] == 'val'])}")
print(f"  - Test       : {len(split_df[split_df['split'] == 'test'])}")

# 4. Liệt kê parts
parts = sorted([d for d in os.listdir(images_base) if d.startswith('part_')])
print(f"\n✓ Image parts tìm thấy: {parts}")

# 5. Lọc theo split cần chạy
if RUN_SPLIT == 'all':
    run_df = split_df
else:
    run_df = split_df[split_df['split'] == RUN_SPLIT]

print(f"\n✓ Clips cần xử lý ({RUN_SPLIT}): {len(run_df)}")
print("\n✓ Tất cả kiểm tra hoàn tất — sẵn sàng chạy tracking!")

KIỂM TRA INPUT
  ✓: Dataset root — /kaggle/input/datasets/kronomy/helmet-dataset-by-osf-lite/helmet-dataset
  ✓: Images base — /kaggle/input/datasets/kronomy/helmet-dataset-by-osf-lite/helmet-dataset/images
  ✓: data_split.csv — /kaggle/input/datasets/kronomy/helmet-dataset-by-osf-lite/helmet-dataset/data_split.csv
  ✓: YOLO weights — /kaggle/input/datasets/uynguyenthai/helmet-step2-outputs/best.pt

✓ Data split:
  - Tổng clips : 910
  - Train      : 636
  - Val        : 92
  - Test       : 182

✓ Image parts tìm thấy: ['part_1', 'part_2']

✓ Clips cần xử lý (test): 182

✓ Tất cả kiểm tra hoàn tất — sẵn sàng chạy tracking!


## Section 4: Tạo botsort_custom.yaml

In [6]:
# ============================================================================
# Tạo file cấu hình BoT-SORT tùy chỉnh
# Lý do:
#   - track_buffer=10: dataset 10 FPS → 1 giây thực tế
#   - with_reid=True : dùng ReID features để tracking tốt hơn
#   - gmc_method=sparseOptFlow: bù trừ camera motion
# ============================================================================

botsort_config = """
# BoT-SORT Custom Configuration (day du, tuong thich Ultralytics >= 8.0)
tracker_type: botsort

# Track buffer (10 FPS x 1 giay = 10 frames)
track_buffer: 10

# Confidence thresholds
track_high_thresh: 0.5
track_low_thresh: 0.1
new_track_thresh: 0.25

# Matching
match_thresh: 0.8

# ReID - tat de tranh loi thieu osnet weights
with_reid: False

# Global Motion Compensation
gmc_method: sparseOptFlow

# IoU-based proximity
proximity_thresh: 0.5
appearance_thresh: 0.25

# fuse_score: bat buoc trong Ultralytics >= 8.1
fuse_score: False
"""

with open(BOTSORT_CFG, 'w') as f:
    f.write(botsort_config)

print(f"✓ Đã tạo file cấu hình BoT-SORT: {BOTSORT_CFG}")
print(f"\nNội dung file:")
print("-" * 40)
print(botsort_config)

✓ Đã tạo file cấu hình BoT-SORT: /kaggle/working/botsort_custom.yaml

Nội dung file:
----------------------------------------

# BoT-SORT Custom Configuration (day du, tuong thich Ultralytics >= 8.0)
tracker_type: botsort

# Track buffer (10 FPS x 1 giay = 10 frames)
track_buffer: 10

# Confidence thresholds
track_high_thresh: 0.5
track_low_thresh: 0.1
new_track_thresh: 0.25

# Matching
match_thresh: 0.8

# ReID - tat de tranh loi thieu osnet weights
with_reid: False

# Global Motion Compensation
gmc_method: sparseOptFlow

# IoU-based proximity
proximity_thresh: 0.5
appearance_thresh: 0.25

# fuse_score: bat buoc trong Ultralytics >= 8.1
fuse_score: False



## Section 5: Định nghĩa TrackManager class

In [7]:
# ============================================================================
# TrackManager: Quản lý vòng đời của các track trong một clip
# 
# Trách nhiệm:
# 1. Accumulation: Thu thập crops cho từng track ID
# 2. Death Trigger: Phát hiện track chết (không xuất hiện > track_buffer frames)
# 3. Handoff: Flush crops xuống disk khi track chết → giải phóng RAM
# 4. Finalize: Flush toàn bộ track còn sót sau khi hết frames
# ============================================================================

class TrackManager:
    """
    Quản lý vòng đời track cho một video clip.
    
    Mỗi clip là một sequence độc lập → phải tạo TrackManager mới cho mỗi clip.
    KHÔNG dùng chung TrackManager giữa các clip!
    """
    
    def __init__(self, video_id, output_dir, track_buffer=10, max_crops=80, 
                 crop_size=224, crop_padding=0.05):
        """
        Khởi tạo TrackManager cho một clip.
        
        Args:
            video_id      : Tên clip (e.g., 'Bago_highway_1')
            output_dir    : Thư mục gốc để lưu crops
            track_buffer  : Số frames tối đa một track có thể 'biến mất'
                            trước khi bị coi là đã kết thúc
            max_crops     : Số crops tối đa được giữ per track
            crop_size     : Kích thước resize crop (pixels, vuông)
            crop_padding  : Tỷ lệ padding mỗi phía (0.05 = 5%)
        """
        self.video_id    = video_id
        self.output_dir  = output_dir
        self.track_buffer = track_buffer
        self.max_crops   = max_crops
        self.crop_size   = crop_size
        self.crop_padding = crop_padding
        
        # Dict lưu crops buffer: {track_id -> list of (frame_idx, crop_img)}
        self.active_tracks = {}
        
        # Dict lưu last seen frame: {track_id -> frame_idx}
        self.last_seen = {}
        
        # Dict lưu metadata của từng track đã flush
        self.track_metadata = []
        
        # Thống kê
        self.total_tracks_processed = 0
        self.total_crops_saved = 0
    
    def _make_crop(self, frame, x1, y1, x2, y2):
        """
        Crop vùng xe máy từ frame, áp dụng padding và resize.
        
        Args:
            frame     : Ảnh gốc (numpy array, BGR)
            x1,y1,x2,y2: Bounding box (pixel coordinates)
        
        Returns:
            Crop đã resize về (crop_size × crop_size) hoặc None nếu lỗi
        """
        h, w = frame.shape[:2]
        
        # Tính padding dựa vào kích thước bbox
        bw = x2 - x1
        bh = y2 - y1
        pad_x = int(bw * self.crop_padding)
        pad_y = int(bh * self.crop_padding)
        
        # Áp dụng padding, clamp vào kích thước frame
        cx1 = max(0, int(x1) - pad_x)
        cy1 = max(0, int(y1) - pad_y)
        cx2 = min(w, int(x2) + pad_x)
        cy2 = min(h, int(y2) + pad_y)
        
        if cx1 >= cx2 or cy1 >= cy2:
            return None
        
        crop = frame[cy1:cy2, cx1:cx2]
        
        if crop.size == 0:
            return None
        
        # Resize về crop_size × crop_size
        crop = cv2.resize(crop, (self.crop_size, self.crop_size), 
                          interpolation=cv2.INTER_LINEAR)
        return crop
    
    def update(self, frame_idx, frame, detections):
        """
        Xử lý kết quả tracking của một frame.
        
        Args:
            frame_idx   : Index frame hiện tại (0-based)
            frame       : Ảnh frame gốc (numpy array, BGR)
            detections  : List các detection từ YOLO tracker,
                          mỗi phần tử là (track_id, x1, y1, x2, y2, conf, cls)
        """
        # Cập nhật active tracks với détections mới
        seen_track_ids = set()
        
        for (track_id, x1, y1, x2, y2, conf, cls) in detections:
            track_id = int(track_id)
            seen_track_ids.add(track_id)
            
            # Khởi tạo track nếu chưa có
            if track_id not in self.active_tracks:
                self.active_tracks[track_id] = []
            
            # Cập nhật last seen
            self.last_seen[track_id] = frame_idx
            
            # Thu thập crop nếu chưa đủ max_crops
            if len(self.active_tracks[track_id]) < self.max_crops:
                crop = self._make_crop(frame, x1, y1, x2, y2)
                if crop is not None:
                    self.active_tracks[track_id].append((frame_idx, crop))
        
        # Kiểm tra track chết: nếu frame_idx - last_seen > track_buffer
        dead_tracks = []
        for track_id, last_f in self.last_seen.items():
            if track_id in self.active_tracks:  # chỉ check track còn active
                if frame_idx - last_f > self.track_buffer:
                    dead_tracks.append(track_id)
        
        # Flush các track đã chết
        for track_id in dead_tracks:
            self._flush_track(track_id)
    
    def _flush_track(self, track_id):
        """
        Lưu toàn bộ crops của một track xuống disk và giải phóng RAM.
        
        Args:
            track_id: ID của track cần flush
        """
        if track_id not in self.active_tracks:
            return
        
        crops = self.active_tracks[track_id]
        
        if len(crops) == 0:
            # Track không có crop nào → bỏ qua
            del self.active_tracks[track_id]
            return
        
        # Tạo thư mục cho track này
        track_dir = os.path.join(self.output_dir, self.video_id, 
                                  f'track_{track_id:04d}')
        os.makedirs(track_dir, exist_ok=True)
        
        # Lưu từng crop
        first_frame = crops[0][0]   if crops else -1
        last_frame  = crops[-1][0]  if crops else -1
        saved_count = 0
        
        for (frame_idx, crop) in crops:
            crop_path = os.path.join(track_dir, f'frame_{frame_idx:06d}.jpg')
            cv2.imwrite(crop_path, crop, 
                        [cv2.IMWRITE_JPEG_QUALITY, 90])
            saved_count += 1
        
        # Lưu metadata cho track này
        self.track_metadata.append({
            'video_id'        : self.video_id,
            'track_id'        : track_id,
            'first_frame'     : first_frame,
            'last_frame'      : last_frame,
            'duration_frames' : last_frame - first_frame + 1,
            'n_crops'         : saved_count,
            'crops_dir'       : track_dir,
        })
        
        self.total_tracks_processed += 1
        self.total_crops_saved += saved_count
        
        # Giải phóng RAM
        del self.active_tracks[track_id]
    
    def finalize(self):
        """
        Flush tất cả các track còn sót lại sau khi hết frames của clip.
        Phải gọi sau vòng lặp frame để đảm bảo không mất data!
        """
        remaining = list(self.active_tracks.keys())
        for track_id in remaining:
            self._flush_track(track_id)
        
        return {
            'total_tracks': self.total_tracks_processed,
            'total_crops' : self.total_crops_saved,
        }

print("✓ TrackManager class đã được định nghĩa")

✓ TrackManager class đã được định nghĩa


## Section 6: Hàm xử lý từng clip

In [8]:
# ============================================================================
# process_clip: Chạy YOLO + BoT-SORT trên một video clip
# 
# Luồng xử lý:
# For each frame in clip:
#   1. Đọc frame từ disk
#   2. Chạy model.track() → lấy bounding boxes + track IDs
#   3. Gọi manager.update() → thu thập crops, phát hiện track chết
# After all frames:
#   4. Gọi manager.finalize() → flush toàn bộ track còn sót
#   5. Reset model.predictor = None → xóa trạng thái tracker
# ============================================================================

def process_clip(model, clip_name, clip_dir, manager, frame_range=(1, 101)):
    """
    Chạy tracking trên một video clip.
    
    Args:
        model       : YOLO model đã load
        clip_name   : Tên clip (e.g., 'Bago_highway_1')
        clip_dir    : Thư mục chứa frames của clip
        manager     : TrackManager instance cho clip này
        frame_range : Tuple (start, end) cho range() của frame index
                      Default: (1, 101) → frames 1.jpg đến 100.jpg
    
    Returns:
        Dict thống kê kết quả của clip
    """
    frames_processed = 0
    frames_with_detections = 0
    errors = []
    
    # QUAN TRỌNG: Phải persist=True để BoT-SORT duy trì track IDs
    # giữa các frame. Nếu không, mỗi frame sẽ có track IDs độc lập!
    
    # Vòng lặp qua từng frame
    for frame_idx in range(*frame_range):
        frame_path = os.path.join(clip_dir, f'{frame_idx}.jpg')
        
        # Bỏ qua frame không tồn tại
        if not os.path.exists(frame_path):
            continue
        
        try:
            # Đọc frame
            frame = cv2.imread(frame_path)
            if frame is None:
                errors.append(f"Frame {frame_idx}: Không đọc được ảnh")
                continue
            
            # Chạy YOLO tracking với persist=True
            # persist=True: BoT-SORT giữ nguyên track IDs giữa các frame
            results = model.track(
                source    = frame,
                conf      = CONFIDENCE_THRESH,
                iou       = IOU_THRESH,
                imgsz     = IMG_SIZE,
                persist   = True,     # DUY TRÌ track IDs giữa frame!
                tracker   = BOTSORT_CFG,
                verbose   = False,
                device    = 0 if torch.cuda.is_available() else 'cpu',
            )
            
            frames_processed += 1
            
            # Trích xuất detections
            detections = []
            for result in results:
                if result.boxes is None:
                    continue
                
                boxes = result.boxes
                
                # Bỏ qua nếu không có track IDs (chưa đủ frames để gán)
                if boxes.id is None:
                    continue
                
                for i in range(len(boxes)):
                    track_id = int(boxes.id[i].item())
                    x1, y1, x2, y2 = boxes.xyxy[i].tolist()
                    conf = boxes.conf[i].item()
                    cls  = int(boxes.cls[i].item())
                    detections.append((track_id, x1, y1, x2, y2, conf, cls))
            
            if detections:
                frames_with_detections += 1
            
            # Cập nhật TrackManager
            manager.update(frame_idx, frame, detections)
            
        except Exception as e:
            errors.append(f"Frame {frame_idx}: {str(e)}")
    
    # BẮT BUỘC: Flush các track còn sót sau khi hết frames
    clip_stats = manager.finalize()
    
    return {
        'clip_name'             : clip_name,
        'frames_processed'      : frames_processed,
        'frames_with_detections': frames_with_detections,
        'total_tracks'          : clip_stats['total_tracks'],
        'total_crops'           : clip_stats['total_crops'],
        'errors'                : errors,
        'track_metadata'        : manager.track_metadata,
    }


def find_clip_dir(clip_name, images_base, parts):
    """
    Tìm thư mục chứa ảnh của một clip trên các parts.
    
    Args:
        clip_name   : Tên clip
        images_base : Thư mục gốc chứa các parts
        parts       : Danh sách tên parts (e.g., ['part_1', 'part_2'])
    
    Returns:
        Đường dẫn đến thư mục clip, hoặc None nếu không tìm thấy
    """
    for part in parts:
        clip_dir = os.path.join(images_base, part, clip_name)
        if os.path.isdir(clip_dir):
            return clip_dir
    return None

print("✓ Hàm process_clip và find_clip_dir đã được định nghĩa")

✓ Hàm process_clip và find_clip_dir đã được định nghĩa


## Section 7: Load YOLO model và chạy tracking

In [9]:
# ============================================================================
# MAIN TRACKING LOOP
# 
# Thiết kế:
# - Mỗi clip tạo TrackManager riêng (KHÔNG dùng chung)
# - Sau mỗi clip: reset model = YOLO(BEST_WEIGHTS)  # FIX: reset an toan, tranh AttributeError để xóa trạng thái BoT-SORT
#   Quan trọng! Nếu không reset, BoT-SORT sẽ nghĩ clips liên tiếp 
#   là cùng một video.
# - Exception handling: log lỗi, bỏ clip lỗi, tiếp tục
# ============================================================================

print("=" * 70)
print("BẮT ĐẦU TRACKING")
print("=" * 70)

start_time = time.time()

# Load YOLO model một lần
print(f"\n[1/3] Loading YOLO model từ {BEST_WEIGHTS}...")
model = YOLO(BEST_WEIGHTS)
print(f"  ✓ Model loaded. Classes: {model.names}")

# Chuẩn bị
parts = sorted([d for d in os.listdir(images_base) if d.startswith('part_')])

# Thống kê tổng
global_stats = {
    'total_clips_processed': 0,
    'total_clips_skipped'  : 0,
    'total_frames'         : 0,
    'total_tracks'         : 0,
    'total_crops'          : 0,
    'clip_stats'           : [],
}

# Danh sách metadata tổng hợp
all_track_metadata = []

print(f"\n[2/3] Chạy tracking trên {len(run_df)} clips ({RUN_SPLIT})...")
print(f"      Tracker config: {BOTSORT_CFG}")
print()

pbar = tqdm(run_df.iterrows(), total=len(run_df), 
            desc="Processing clips", unit="clip")

for _, row in pbar:
    clip_name = row['video_id']
    split     = row['split']
    
    pbar.set_postfix(clip=clip_name[:20])
    
    # Tìm thư mục ảnh của clip
    clip_dir = find_clip_dir(clip_name, images_base, parts)
    if clip_dir is None:
        global_stats['total_clips_skipped'] += 1
        tqdm.write(f"  ⚠️  Bỏ qua {clip_name}: không tìm thấy thư mục ảnh")
        continue
    
    try:
        # QUAN TRỌNG: Reset tracker trước mỗi clip mới!
        # Dùng model = YOLO(BEST_WEIGHTS)  # FIX: reset an toan, tranh AttributeError gây lỗi AttributeError khi BoT-SORT chưa
        # warmup đầy đủ → phải khởi tạo lại YOLO instance để reset an toàn.
        model = YOLO(BEST_WEIGHTS)
        
        # Tạo TrackManager mới cho clip này (KHÔNG dùng lại)
        manager = TrackManager(
            video_id     = clip_name,
            output_dir   = TRACK_CROPS_DIR,
            track_buffer = TRACK_BUFFER,
            max_crops    = MAX_CROPS_PER_TRACK,
            crop_size    = CROP_SIZE,
            crop_padding = CROP_PADDING,
        )
        
        # Chạy tracking trên clip này
        clip_result = process_clip(
            model      = model,
            clip_name  = clip_name,
            clip_dir   = clip_dir,
            manager    = manager,
            frame_range = (1, 101),  # frames 1.jpg đến 100.jpg
        )
        
        # Thêm split info vào metadata
        for meta in clip_result['track_metadata']:
            meta['split'] = split
        all_track_metadata.extend(clip_result['track_metadata'])
        
        # Cập nhật thống kê
        global_stats['total_clips_processed'] += 1
        global_stats['total_frames']   += clip_result['frames_processed']
        global_stats['total_tracks']   += clip_result['total_tracks']
        global_stats['total_crops']    += clip_result['total_crops']
        global_stats['clip_stats'].append({
            'clip_name'    : clip_name,
            'split'        : split,
            'frames'       : clip_result['frames_processed'],
            'tracks'       : clip_result['total_tracks'],
            'crops'        : clip_result['total_crops'],
            'n_errors'     : len(clip_result['errors']),
        })
        
        # Log lỗi (nếu có) nhưng không dừng pipeline
        if clip_result['errors']:
            tqdm.write(f"  ⚠️  {clip_name}: {len(clip_result['errors'])} lỗi frame")
            for err in clip_result['errors'][:3]:  # chỉ log 3 lỗi đầu
                tqdm.write(f"       {err}")
    
    except Exception as e:
        global_stats['total_clips_skipped'] += 1
        tqdm.write(f"  ✗ Lỗi clip {clip_name}: {str(e)}")

elapsed = time.time() - start_time

print(f"\n[3/3] Lưu metadata...")

# Lưu track metadata
with open(METADATA_PATH, 'w') as f:
    json.dump(all_track_metadata, f, indent=2)
print(f"  ✓ track_metadata.json: {len(all_track_metadata)} tracks")

# Lưu summary
global_stats['elapsed_seconds'] = elapsed
global_stats['elapsed_human']   = f"{elapsed/3600:.1f}h {(elapsed%3600)/60:.0f}m"
# Loại bỏ clip_stats chi tiết để JSON nhỏ hơn
summary_save = {k: v for k, v in global_stats.items() if k != 'clip_stats'}
summary_save['split_run'] = RUN_SPLIT
with open(SUMMARY_PATH, 'w') as f:
    json.dump(summary_save, f, indent=2)
print(f"  ✓ step3_tracking_summary.json saved")

print("\n" + "=" * 70)
print("✓ TRACKING HOÀN THÀNH!")
print(f"  Split chạy         : {RUN_SPLIT}")
print(f"  Clips đã xử lý    : {global_stats['total_clips_processed']}")
print(f"  Clips bị bỏ qua   : {global_stats['total_clips_skipped']}")
print(f"  Tổng frames        : {global_stats['total_frames']:,}")
print(f"  Tổng tracks        : {global_stats['total_tracks']:,}")
print(f"  Tổng crops         : {global_stats['total_crops']:,}")
print(f"  Thời gian chạy     : {elapsed/3600:.1f}h {(elapsed%3600)/60:.0f}m")
print("=" * 70)

BẮT ĐẦU TRACKING

[1/3] Loading YOLO model từ /kaggle/input/datasets/uynguyenthai/helmet-step2-outputs/best.pt...
  ✓ Model loaded. Classes: {0: 'compliant', 1: 'violation'}

[2/3] Chạy tracking trên 182 clips (test)...
      Tracker config: /kaggle/working/botsort_custom.yaml



Processing clips:  22%|██▏       | 40/182 [04:39<17:08,  7.24s/clip, clip=Mandalay_1_18] 

  ⚠️  Bỏ qua Mandalay_1_139: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Mandalay_1_140: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Mandalay_1_143: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Mandalay_1_148: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Mandalay_1_152: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Mandalay_1_155: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Mandalay_1_161: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Mandalay_1_163: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Mandalay_1_166: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Mandalay_1_171: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Mandalay_1_173: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Mandalay_1_177: không tìm thấy thư mục ảnh


Processing clips:  29%|██▉       | 53/182 [04:46<03:21,  1.56s/clip, clip=Mandalay_1_20] 

  ⚠️  Bỏ qua Mandalay_1_182: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Mandalay_1_185: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Mandalay_1_187: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Mandalay_1_190: không tìm thấy thư mục ảnh


...

  ⚠️  Bỏ qua Mandalay_2_108: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Mandalay_2_113: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Mandalay_2_115: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Mandalay_2_117: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Mandalay_2_123: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Mandalay_2_124: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Mandalay_2_126: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Mandalay_2_130: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Mandalay_2_132: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Mandalay_2_137: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Mandalay_2_139: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Mandalay_2_14: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Mandalay_2_148: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Mandalay_2_152: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Mandalay_2_154: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Mandalay_2_156: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Mandalay_2_159: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Mandalay_2_164: không tìm thấy thư m

Processing clips:  79%|███████▉  | 144/182 [06:30<00:06,  6.25clip/s, clip=Pakokku_urban_59]

  ⚠️  Bỏ qua Naypyitaw_2_34: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Naypyitaw_2_35: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Naypyitaw_2_4: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Naypyitaw_2_5: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Naypyitaw_2_8: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Naypyitaw_2_9: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua NyaungU_rural_11: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua NyaungU_rural_14: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua NyaungU_rural_24: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua NyaungU_rural_27: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua NyaungU_rural_28: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua NyaungU_rural_32: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua NyaungU_rural_33: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua NyaungU_rural_39: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua NyaungU_rural_41: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua NyaungU_rural_56: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua NyaungU_rural_58: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua NyaungU_rural_63:

Processing clips: 100%|██████████| 182/182 [06:31<00:00,  2.15s/clip, clip=Yangon_II_6]


  ⚠️  Bỏ qua Pakokku_urban_59: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Pakokku_urban_62: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Pakokku_urban_69: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Pakokku_urban_71: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Pathein_rural_2: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Pathein_rural_6: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Pathein_rural_7: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Yangon_II_14: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Yangon_II_27: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Yangon_II_28: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Yangon_II_29: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Yangon_II_30: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Yangon_II_32: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Yangon_II_33: không tìm thấy thư mục ảnh
  ⚠️  Bỏ qua Yangon_II_6: không tìm thấy thư mục ảnh

[3/3] Lưu metadata...
  ✓ track_metadata.json: 451 tracks
  ✓ step3_tracking_summary.json saved

✓ TRACKING HOÀN THÀNH!
  Split chạy         : test
  Clips đã xử lý 

## Section 8: Kiểm tra Output

In [10]:
# ============================================================================
# Kiểm tra và thống kê kết quả output
# ============================================================================

print("=" * 70)
print("KIỂM TRA OUTPUT")
print("=" * 70)

# 1. Đọc lại metadata
with open(METADATA_PATH, 'r') as f:
    metadata = json.load(f)

print(f"\n1. METADATA TỔNG QUAN:")
print(f"   Tổng số tracks    : {len(metadata)}")

if metadata:
    # Phân tích phân phối
    n_crops_list = [m['n_crops'] for m in metadata]
    duration_list = [m['duration_frames'] for m in metadata]
    
    print(f"\n2. PHÂN PHỐI CROPS PER TRACK:")
    print(f"   Min  : {min(n_crops_list)}")
    print(f"   Max  : {max(n_crops_list)}")
    print(f"   Mean : {sum(n_crops_list)/len(n_crops_list):.1f}")
    print(f"   Total: {sum(n_crops_list):,}")
    
    print(f"\n3. PHÂN PHỐI DURATION (frames):")
    print(f"   Min  : {min(duration_list)}")
    print(f"   Max  : {max(duration_list)}")
    print(f"   Mean : {sum(duration_list)/len(duration_list):.1f}")
    
    # Phân tích theo split (nếu có)
    if 'split' in metadata[0]:
        from collections import Counter
        split_track_counts = Counter(m['split'] for m in metadata)
        split_crop_sums = defaultdict(int)
        for m in metadata:
            split_crop_sums[m['split']] += m['n_crops']
        
        print(f"\n4. PHÂN CHIA THEO SPLIT:")
        for spl in ['train', 'val', 'test']:
            if spl in split_track_counts:
                print(f"   {spl:5s}: {split_track_counts[spl]:4d} tracks, "
                     f"{split_crop_sums[spl]:6,} crops")

# 2. Kiểm tra cấu trúc thư mục
print(f"\n5. CẤU TRÚC OUTPUT:")
if os.path.exists(TRACK_CROPS_DIR):
    video_dirs = os.listdir(TRACK_CROPS_DIR)
    print(f"   track_crops/ : {len(video_dirs)} video directories")
    
    # Kiểm tra sample
    if video_dirs:
        sample_video = video_dirs[0]
        sample_video_dir = os.path.join(TRACK_CROPS_DIR, sample_video)
        track_dirs = os.listdir(sample_video_dir)
        print(f"   Sample ({sample_video}): {len(track_dirs)} tracks")
        
        if track_dirs:
            sample_track = track_dirs[0]
            sample_track_dir = os.path.join(sample_video_dir, sample_track)
            crops = os.listdir(sample_track_dir)
            print(f"   Sample track ({sample_track}): {len(crops)} crops")
            if crops:
                sample_crop = os.path.join(sample_track_dir, crops[0])
                crop_img = cv2.imread(sample_crop)
                if crop_img is not None:
                    print(f"   Sample crop size: {crop_img.shape}  (H×W×C)")

print(f"\n✓ Output saved tại: {OUTPUT_BASE}")
print(f"   - track_crops/              : crops theo track ID")
print(f"   - track_metadata.json       : metadata từng track")
print(f"   - step3_tracking_summary.json: thống kê tổng")
print(f"   - botsort_custom.yaml       : config BoT-SORT đã dùng")
print(f"\n✓ Sẵn sàng sử dụng ở Bước 5 (End-to-End Pipeline)!")

KIỂM TRA OUTPUT

1. METADATA TỔNG QUAN:
   Tổng số tracks    : 451

2. PHÂN PHỐI CROPS PER TRACK:
   Min  : 1
   Max  : 80
   Mean : 29.6
   Total: 13,354

3. PHÂN PHỐI DURATION (frames):
   Min  : 1
   Max  : 91
   Mean : 30.7

4. PHÂN CHIA THEO SPLIT:
   test :  451 tracks, 13,354 crops

5. CẤU TRÚC OUTPUT:
   track_crops/ : 55 video directories
   Sample (Bago_highway_9): 7 tracks
   Sample track (track_0006): 18 crops
   Sample crop size: (224, 224, 3)  (H×W×C)

✓ Output saved tại: /kaggle/working
   - track_crops/              : crops theo track ID
   - track_metadata.json       : metadata từng track
   - step3_tracking_summary.json: thống kê tổng
   - botsort_custom.yaml       : config BoT-SORT đã dùng

✓ Sẵn sàng sử dụng ở Bước 5 (End-to-End Pipeline)!


## Ghi chú & Bước tiếp theo

### Bước 3 hoàn thành ✓

**Output của Bước 3:**
- `track_crops/` — Crops xe máy theo từng Track ID (predicted, không phải GT)
- `track_metadata.json` — Metadata: `{track_id, video_id, first_frame, last_frame, duration_frames, n_crops, crops_dir, split}`
- `step3_tracking_summary.json` — Thống kê tổng (số tracks, crops, thời gian)
- `botsort_custom.yaml` — File config BoT-SORT đã dùng

### Lưu ý quan trọng

| | Bước 3 (output này) | Bước 1 (GT crops) |
|---|---|---|
| **Dùng cho** | Bước 5 (End-to-End Pipeline) | Bước 4 (Train EfficientNetV2) |
| **Track ID** | Predicted (BoT-SORT) | Ground Truth (annotation CSV) |
| **Label** | Không biết (inference) | Đã biết (DHelmet, DNoHelmet...) |

### Bước tiếp theo

1. **Bước 4** (Notebook riêng): Train EfficientNetV2 dùng output **Bước 1** (GT crops)
2. **Bước 5** (Notebook riêng): Kết hợp output Bước 3 + model Bước 4 → CHUE Voting
3. **Bước 6**: Đánh giá kết quả Bước 5 so với Ground Truth

### Hướng dẫn upload output lên Kaggle

Sau khi notebook chạy xong:
1. Download `track_crops/`, `track_metadata.json`, `step3_tracking_summary.json` từ `/kaggle/working/`
2. Tạo Kaggle Dataset mới: `helmet-step3-outputs`
3. Upload các file trên vào dataset
4. Thêm dataset làm input cho Notebook Bước 5

**Lưu ý:** `track_crops/` có thể rất lớn (vài GB). Có thể dùng Kaggle Output để không phải download/upload thủ công.